# Phase 3 — Real-Data Verification Gate

**Purpose: dataset validation, not model results.** Do not train anything in this notebook.

This notebook implements the exact gate agreed on for Phase 3: install the pinned
research environment, run the 185-test suite for real, attempt the real UPFD
download, and inspect the actual dataset against every assumption the scaffold's
code currently rests on — reporting each finding as one of:

1. **directly observed** from the downloaded dataset
2. **confirmed** from project documentation
3. **inferred**
4. **still unresolved**

If any assumption the current architecture depends on is contradicted by the real
data, **stop and report it** — do not patch code just to make something run.

Upload `propagate_phase3_baseline.zip` to this Colab session's file browser before
running the first cell, or mount Drive and point `ZIP_PATH` at it there.

In [ ]:
ZIP_PATH = "/content/propagate_phase3_baseline.zip"  # adjust if uploaded elsewhere

import zipfile, os
assert os.path.exists(ZIP_PATH), f"Upload the zip first -- not found at {ZIP_PATH}"
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("/content/propagate")
os.chdir("/content/propagate/research")
print("Working directory:", os.getcwd())
!ls


## Step 1 — Install exactly `research/requirements.txt`

No extra packages, nothing upgraded silently.

In [ ]:
!pip install -q -r requirements.txt
import torch, torch_geometric, sentence_transformers, sklearn, numpy, scipy, networkx, yaml
print("torch:", torch.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())


## Step 2 — Run the 185-test suite for real, in this environment

This is a re-verification, not a formality: the sandbox that built this scaffold had a different torch build (CUDA-tagged, CPU-only execution) — confirming the same 185 pass here, on Colab's actual runtime, is a genuine check, not a repeat.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q"],
    cwd=".", env={**os.environ, "PYTHONPATH": "propagate_research:."},
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "STOP: tests do not pass in this environment -- do not proceed to real data until this is fixed"
print("\n185/185 CONFIRMED in this Colab environment.")


## Step 3 — Attempt the real UPFD download

Uses the project's own loader (`data/upfd_loader.py`), not a hand-rolled download -- if the loader itself is wrong, that should surface here, not be worked around.

If PyG's automatic downloader fails (a known, previously-documented open issue: Google Drive sometimes returns a virus-scan-warning page instead of the archive for this exact dataset), see `manual_download_instructions()`'s output for the fallback before assuming the dataset itself is broken.

In [ ]:
import sys
sys.path.insert(0, "propagate_research")
from propagate_research.data.upfd_loader import UPFDLoadConfig, load_upfd_split, manual_download_instructions

DATASET_ROOT = "/content/upfd_data"
DOMAIN = "politifact"  # run gossipcop separately below once this domain is verified

config = UPFDLoadConfig(root=DATASET_ROOT, name=DOMAIN, feature="bert")

try:
    train_ds = load_upfd_split(config, "train")
    val_ds = load_upfd_split(config, "val")
    test_ds = load_upfd_split(config, "test")
    print(f"OBSERVED: {DOMAIN} loaded -- train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")
except Exception as e:
    print("Automatic download failed:", type(e).__name__, e)
    print()
    print(manual_download_instructions(config))
    raise


## Step 4 — Inspect the real dataset against every assumption

Each check below prints its finding tagged `[OBSERVED]`, `[DOC]`, `[INFERRED]`, or `[UNRESOLVED]`. Read the printed verdicts before continuing to Step 5 --  if anything prints `CONTRADICTION`, stop here and report it rather than continuing.

In [ ]:
# --- graph counts, per split, per domain ---
print(f"[OBSERVED] {DOMAIN} train graphs: {len(train_ds)}")
print(f"[OBSERVED] {DOMAIN} val graphs:   {len(val_ds)}")
print(f"[OBSERVED] {DOMAIN} test graphs:  {len(test_ds)}")
total = len(train_ds) + len(val_ds) + len(test_ds)
print(f"[OBSERVED] {DOMAIN} total graphs: {total}")

# Published count from the UPFD paper (arXiv:2104.12259), Table 1 --
# used only as a plausibility check, per upfd_loader.py's own PUBLISHED_GRAPH_COUNTS
from propagate_research.data.upfd_loader import PUBLISHED_GRAPH_COUNTS
expected_total = PUBLISHED_GRAPH_COUNTS[DOMAIN]["total"]
print(f"[DOC] Published total for {DOMAIN}: {expected_total}")
if total != expected_total:
    print(f"CONTRADICTION: observed total {total} != published {expected_total} -- investigate before proceeding")
else:
    print("[OBSERVED] matches published total")


In [ ]:
# --- node counts, feature dimensions/dtypes, root-node indexing ---
sample = train_ds[0]
print(f"[OBSERVED] sample graph: num_nodes={sample.num_nodes}")
print(f"[OBSERVED] x.shape={tuple(sample.x.shape)}, x.dtype={sample.x.dtype}")
print(f"[OBSERVED] edge_index.shape={tuple(sample.edge_index.shape)}, dtype={sample.edge_index.dtype}")
print(f"[OBSERVED] y={sample.y}")

# root/leaf feature-width consistency -- the assumption the whole
# dual-relation design depends on (see research/README.md)
root_width = sample.x[0].shape[0]
all_same_width = all(sample.x[i].shape[0] == root_width for i in range(sample.num_nodes))
print(f"[OBSERVED] root node (index 0) feature width: {root_width}")
print(f"[OBSERVED] every node shares that width: {all_same_width}")
if not all_same_width:
    print("CONTRADICTION: root/leaf feature-width mismatch -- the dual-relation "
          "GNN as currently built cannot consume this graph. STOP.")


In [ ]:
# --- edge direction, direct-share/inherited-share derivation ---
from propagate_research.graph.relations import (
    split_edges_by_relation, validate_root_first_convention, ROOT_NODE_INDEX
)

edge_list = [tuple(e) for e in sample.edge_index.t().tolist()]
try:
    validate_root_first_convention(edge_list, num_nodes=sample.num_nodes)
    print("[OBSERVED] root-first cascade convention holds for this sample graph")
except ValueError as e:
    print(f"CONTRADICTION: {e}")

split = split_edges_by_relation(edge_list)
print(f"[OBSERVED] direct_share edges: {len(split.direct_share)}")
print(f"[OBSERVED] inherited_share edges: {len(split.inherited_share)}")
print(f"[OBSERVED] total edges accounted for: {len(split.all_edges)} / {len(edge_list)}")

# Check the assumption across a larger sample, not just graph 0
import random
random.seed(0)
sample_indices = random.sample(range(len(train_ds)), min(50, len(train_ds)))
violations = []
for i in sample_indices:
    g = train_ds[i]
    e_list = [tuple(e) for e in g.edge_index.t().tolist()]
    try:
        validate_root_first_convention(e_list, num_nodes=g.num_nodes)
    except ValueError as e:
        violations.append((i, str(e)))
print(f"[OBSERVED] root-first convention checked on {len(sample_indices)} random graphs: "
      f"{len(violations)} violations")
if violations:
    print("CONTRADICTION -- sample violations:", violations[:5])


In [ ]:
# --- labels and actual polarity ---
from propagate_research.data.upfd_loader import assert_label_polarity_plausible

all_labels = [int(train_ds[i].y.item()) for i in range(len(train_ds))]
ones = sum(all_labels)
zeros = len(all_labels) - ones
print(f"[OBSERVED] train split: {ones} labeled 1, {zeros} labeled 0")

try:
    assert_label_polarity_plausible(all_labels, DOMAIN)
    print("[OBSERVED] label polarity plausible against published fake/real counts "
          "(does not prove which polarity is correct -- see upfd_loader.py docstring)")
except ValueError as e:
    print(f"CONTRADICTION: {e}")


In [ ]:
# --- feature dimensions for all four feature types (not just 'bert') ---
for feature in ["profile", "spacy", "bert", "content"]:
    try:
        cfg = UPFDLoadConfig(root=DATASET_ROOT, name=DOMAIN, feature=feature)
        ds = load_upfd_split(cfg, "train")
        s = ds[0]
        print(f"[OBSERVED] feature={feature}: x.shape={tuple(s.x.shape)}")
    except Exception as e:
        print(f"[OBSERVED] feature={feature}: failed to load -- {type(e).__name__}: {e}")


In [ ]:
# --- duplicate / missing / invalid value checks ---
import torch as _torch

nan_count = sum(1 for i in range(len(train_ds)) if _torch.isnan(train_ds[i].x).any())
inf_count = sum(1 for i in range(len(train_ds)) if _torch.isinf(train_ds[i].x).any())
empty_edge_count = sum(1 for i in range(len(train_ds)) if train_ds[i].edge_index.shape[1] == 0)
print(f"[OBSERVED] graphs with NaN in x: {nan_count} / {len(train_ds)}")
print(f"[OBSERVED] graphs with Inf in x: {inf_count} / {len(train_ds)}")
print(f"[OBSERVED] graphs with zero edges (root-only): {empty_edge_count} / {len(train_ds)}")

# duplicate graph_id check across splits, if the raw dataset exposes one
# (this is the leakage-adjacent check we can actually run without ground-truth
# article IDs -- true article-level dedup would need FakeNewsNet's own IDs,
# which UPFD's public release does not expose; flag that limitation explicitly)
print("[UNRESOLVED] Article-level duplicate/leakage detection across domains "
      "requires original FakeNewsNet article IDs, which UPFD's public graph "
      "release does not expose -- cannot be checked from this data alone.")


## Step 5 — GossipCop: repeat the same checks on the second domain

Don't assume PolitiFact's results generalize to GossipCop -- it's a much larger, more imbalanced dataset per the published counts.

In [ ]:
DOMAIN = "gossipcop"
config = UPFDLoadConfig(root=DATASET_ROOT, name=DOMAIN, feature="bert")
train_ds = load_upfd_split(config, "train")
val_ds = load_upfd_split(config, "val")
test_ds = load_upfd_split(config, "test")

total = len(train_ds) + len(val_ds) + len(test_ds)
expected_total = PUBLISHED_GRAPH_COUNTS[DOMAIN]["total"]
print(f"[OBSERVED] {DOMAIN} total graphs: {total} (published: {expected_total})")

all_labels = [int(train_ds[i].y.item()) for i in range(len(train_ds))]
try:
    assert_label_polarity_plausible(all_labels, DOMAIN)
    print("[OBSERVED] gossipcop label polarity plausible")
except ValueError as e:
    print(f"CONTRADICTION: {e}")

sample = train_ds[0]
root_width = sample.x[0].shape[0]
all_same_width = all(sample.x[i].shape[0] == root_width for i in range(sample.num_nodes))
print(f"[OBSERVED] gossipcop feature width uniform across nodes: {all_same_width}")


## Step 6 — Verdict

Fill in after reading every cell's output above:

| Assumption | Status |
|---|---|
| Graph counts match published totals | |
| Root/leaf feature-width uniform | |
| Root-first cascade convention holds | |
| direct-share/inherited-share derivation valid | |
| Label polarity plausible | |
| No NaN/Inf in features | |

**If every row is a genuine pass:** proceed to Step 7 (encoder verification) below,
still not training.

**If any row contradicts the scaffold's assumptions:** stop here. Bring the specific
finding back before changing any code — the smallest scientifically defensible
change gets decided there, not improvised in this notebook.

## Step 7 — Encoder verification (only after Step 6 passes)

Confirms `encoders/content_encoder.py` actually works end-to-end now that huggingface.co is reachable -- this could never be verified in the original sandbox.

In [ ]:
from propagate_research.encoders.content_encoder import ContentEncoder

encoder = ContentEncoder()  # default: all-MiniLM-L6-v2
texts = ["A real news article about the economy.", "A completely different sentence."]
emb = encoder.encode(texts)
print(f"[OBSERVED] embedding shape: {emb.shape}")
print(f"[OBSERVED] embedding_dim property: {encoder.embedding_dim}")

# determinism check
emb2 = encoder.encode(texts)
import numpy as np
print(f"[OBSERVED] deterministic across calls: {np.allclose(emb, emb2)}")


## Do not proceed past this notebook to training

This notebook's job ends at dataset + encoder validation. Bring the Step 6 verdict table and any CONTRADICTION findings back for review before writing or running any training code.